# 14 — Error Handling & Resources

Two threads have been running side-by-side through this whole track. **Expected failure** — the kinds you've modelled as `Option`, `Try`, and `Either` in notebook 09. And **unexpected failure** — the kinds the JVM raises as `Throwable`s when something genuinely goes wrong: a network drop, a corrupted file, a bug. This notebook draws a line between them, then adds the missing piece: how to clean up *resources* — open files, sockets, database connections — in the presence of either kind of failure.

Three sets of tools to learn:

- **`throw` / `try` / `catch` / `finally`** — the JVM mechanism that sits underneath everything else. Scala inherits Java's exception model essentially unchanged.
- **`scala.util.Using`** — the standard-library equivalent of Java's try-with-resources. It guarantees cleanup and lifts the result into a `Try`.
- **Patterns for async** — what changes once `Future` is in the picture, and where you need to reach for `IO` or `ZIO` instead.

By the end you should know, for any given may-fail operation: should I model it with `Either`, wrap it in `Try`, let the exception propagate, or move to an effect system?

## `throw` and `try`/`catch` — the JVM mechanism

Every JVM language inherits a single error-signalling primitive: a thrown `Throwable` unwinds the stack until a matching `catch` handles it, or it reaches the top and crashes the thread. Scala uses exactly that mechanism. `throw` is an expression of type `Nothing` — it never returns a value, so the compiler is happy to let it stand in for any expected type. That's why you can write `if cond then throw ... else value` and the whole thing still type-checks as the type of `value`.

In [ ]:
def divide(a: Int, b: Int): Int =
  if b == 0 then throw new IllegalArgumentException("b must be non-zero")
  else a / b

val r =
  try divide(10, 0)
  catch
    case e: IllegalArgumentException => -1
// r: Int = -1

A `try`/`catch` in Scala is itself an *expression* — the `try` block and each `catch` branch produce a value, and the whole construct returns one of them. That's why `val r = try ... catch ...` works. The `catch` branches are pattern matches, exactly like a `match` — specific exception types, name bindings, and guards all behave the same way.

## Catch only what's safe — `NonFatal`

A bare `case _: Throwable =>` is almost always wrong. It will swallow `OutOfMemoryError`, `StackOverflowError`, and `InterruptedException` — errors that mean the JVM or your thread is genuinely in trouble and should propagate. The convention is to use `scala.util.control.NonFatal` as the pattern; it filters out the fatal cases for you.

In [ ]:
import scala.util.control.NonFatal

def safeRun[A](body: => A): Either[Throwable, A] =
  try Right(body)
  catch case NonFatal(e) => Left(e)

safeRun(10 / 2)        // Right(5)
safeRun(10 / 0)        // Left(java.lang.ArithmeticException: / by zero)
// safeRun(throw new OutOfMemoryError) — NonFatal does not catch this; it propagates

Note the `body: => A` parameter — it's *by-name*, so the expression isn't evaluated until `safeRun` calls it inside the try. Without by-name, the exception would fire on the *caller's* line, never reaching the try.

This is the same trick `Try(...)` uses internally. In fact, `Try` is essentially `safeRun` with a richer API. Prefer `Try` for one-off captures; reach for raw `try`/`catch` when you need multiple catch clauses, want to return shapes other than `Try`, or are interoperating with code that throws specific exception types you want to handle differently.

## `try`/`finally` — cleanup on every path

A `finally` block runs whether the `try` succeeded, threw, or a `catch` branch itself threw. Use it to release resources — close a file, return a connection to the pool — that *must* be released regardless of outcome.

In [ ]:
import java.io.{BufferedReader, FileReader}

def firstLine(path: String): String =
  val reader = new BufferedReader(new FileReader(path))
  try reader.readLine()
  finally reader.close()

That's the classic pattern. The reader is opened *before* the try so that a failure during construction doesn't slip past the cleanup, and it's closed in `finally` so a `readLine` failure still releases the file descriptor.

Three reasons not to write this by hand, though:

- **An exception in `finally` wins.** If `readLine` throws *and then* `close` also throws, only the second exception escapes — the original cause is silently lost. This is a real source of misleading stack traces in production.
- **One resource per nesting level.** Two resources mean two `try`/`finally` pairs, each with its own setup and teardown. The boilerplate compounds quickly.
- **Easy to forget.** Any future edit can introduce an early return or a new throw between open and close. The cleanup is not enforced by the compiler.

The fix is `Using`.

## `scala.util.Using` — the resource pattern

`Using.apply` takes a resource and a body that uses it. It guarantees the resource is closed when the body finishes — success or failure — and returns a `Try[A]` carrying the body's result (or any exception, with a suppressed close exception attached if one also occurred). It is the Scala-idiomatic equivalent of Java's try-with-resources.

In [ ]:
import scala.util.{Using, Try}
import java.io.{BufferedReader, FileReader}

def firstLine(path: String): Try[String] =
  Using(new BufferedReader(new FileReader(path))) { reader =>
    reader.readLine()
  }

firstLine("/etc/hosts")     // Success("...")
firstLine("/no/such")       // Failure(java.io.FileNotFoundException: ...)

Two things to notice. First, the body's result is wrapped in `Try` — the same shape from notebook 09, so all of `map`, `flatMap`, `recover`, and the `for` comprehension work on it directly. Second, you didn't write `close` anywhere. `Using` looks up a typeclass instance that knows how to release the resource and calls it for you.

The typeclass is `Using.Releasable[R]`. The standard library already supplies an instance for anything that extends `java.lang.AutoCloseable` — which includes most JDK I/O types — so you usually get it for free.

## Custom resources

If your resource doesn't already implement `AutoCloseable`, you have two choices: make it implement `AutoCloseable`, or provide a `Releasable` given. Both are short.

In [ ]:
import scala.util.Using

// Option 1 — extend AutoCloseable
class Session(name: String) extends AutoCloseable:
  println(s"opened $name")
  def query(s: String): String = s"<$name: $s>"
  def close(): Unit = println(s"closed $name")

Using(new Session("db")) { s => s.query("select 1") }
// prints: opened db
//         closed db
// res: Try[String] = Success(<db: select 1>)

// Option 2 — provide a Releasable given for a type you don't control
class Cache(name: String):
  def get(k: String): Option[String] = Some(s"$name/$k")
  def shutdown(): Unit = println(s"shutdown $name")

given Using.Releasable[Cache] with
  def release(c: Cache): Unit = c.shutdown()

Using(Cache("hot")) { c => c.get("k") }
// prints: shutdown hot
// res: Try[Option[String]] = Success(Some(hot/k))

The first form is the right default. The `Releasable` given is what you reach for when you can't modify the resource type — typically a third-party class that doesn't extend `AutoCloseable` but does have some other "stop" or "shutdown" method.

## Multiple resources — `Using.Manager`

Real code often opens several resources at once: a database connection plus a result-set reader, an HTTP client plus a response stream. Nesting `Using.apply` calls works for two resources and stops being readable at three. `Using.Manager` flattens the nesting — you register each resource with the manager as you acquire it, and the manager releases them all in reverse order when the block ends.

In [ ]:
import scala.util.{Using, Try}
import java.io.{BufferedReader, FileReader, PrintWriter, FileWriter}

def copyFirstLine(src: String, dst: String): Try[Unit] =
  Using.Manager { use =>
    val in  = use(new BufferedReader(new FileReader(src)))
    val out = use(new PrintWriter(new FileWriter(dst)))
    out.println(in.readLine())
  }

Each `use(...)` call registers the resource and returns it. When the block exits — for any reason — the manager calls `close` on each in LIFO order. If both a body exception and a close exception occur, the body exception wins and the close exception is attached as suppressed, so you don't lose information the way you do with hand-written `try`/`finally`.

## Mixing `Using` with your own error type

`Using` returns a `Try`, which means failure is a `Throwable`. Often you'd rather expose named errors to callers, as in notebook 09. The translation is one `.toEither` followed by `.left.map` to convert the throwable into your domain error.

In [ ]:
import scala.util.Using
import java.io.{BufferedReader, FileReader}

enum LoadError:
  case NotFound(path: String)
  case Unreadable(path: String, cause: Throwable)

def loadFirstLine(path: String): Either[LoadError, String] =
  Using(new BufferedReader(new FileReader(path)))(_.readLine())
    .toEither
    .left.map {
      case _: java.io.FileNotFoundException => LoadError.NotFound(path)
      case other                            => LoadError.Unreadable(path, other)
    }

Read that as the bridge between the two error worlds. Inside, a `Try` from `Using` — capture-shaped, throwable-typed. Outside, an `Either[LoadError, String]` — domain-shaped, enum-typed. Every caller now has to handle `NotFound` and `Unreadable` explicitly, and adding a third case to `LoadError` would force the compiler to flag every match downstream.

## Async — resources plus `Future`

`Using` is synchronous. The block must produce its result *before* the resource is closed, which means if your block returns a `Future[A]` the resource is closed before the future completes — and the asynchronous read will fail or return garbage. Two patterns avoid the trap.

- Keep the I/O inside the `Future` block, so the resource lives and dies on one async thread.
- Use `Future#andThen` (or `transform`) to register cleanup that runs when the future completes.

In [ ]:
import scala.concurrent.{Future, ExecutionContext}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.util.Using
import java.io.{BufferedReader, FileReader}

// Pattern 1 — resource is acquired, used, and released inside the Future block.
def asyncFirstLine(path: String): Future[String] =
  Future {
    Using(new BufferedReader(new FileReader(path)))(_.readLine()).get
  }

// Pattern 2 — manual cleanup tied to the Future's lifecycle.
def asyncLineCount(path: String): Future[Int] =
  val reader = new BufferedReader(new FileReader(path))
  Future {
    Iterator.continually(reader.readLine()).takeWhile(_ != null).size
  }.andThen { case _ => reader.close() }

Pattern 1 is the right default — short-lived resource, contained lifetime. Pattern 2 is what you reach for when the resource is shared across multiple async steps; the `andThen` callback runs on completion regardless of success or failure.

Neither is perfect. There is no built-in equivalent of `Using.Manager` for `Future`. If you find yourself stacking these patterns, that's the signal to move to `cats-effect`'s `Resource[IO, A]` or ZIO's `Scope` — both give you composable, leak-safe resource management with the same `for`-comprehension feel as everything else in notebooks 09 and 13.

## Throw vs return — a working rule of thumb

You now have five ways to surface a failure. The choice is rarely arbitrary.

- **Return `Option[A]`** when absence is the only failure mode and no reason is needed.
- **Return `Try[A]`** when wrapping a call that throws and the caller just wants value-or-exception.
- **Return `Either[E, A]`** when failures are part of your domain and the caller must distinguish them.
- **Throw an exception** when the failure is a *bug* — a precondition violation, a "should never happen" — or you're at a boundary that uses exceptions (a Java callback, a servlet, an actor's `receive`).
- **Let it propagate** when you can't do anything useful with the failure at this layer. Don't write empty catch blocks; let a boundary handler deal with it.

The bias should be toward returning values, not throwing. Throwing makes a failure invisible in the type signature. A function that returns `Either[E, A]` cannot be misused — the compiler forces the caller to address `E`. A function that throws can be misused on every call site.

## Common pitfalls

The mistakes that catch everyone at least once.

- **Catching `Throwable`.** Use `NonFatal`, or specific exception types. A bare `Throwable` swallows JVM-level errors that need to crash the process.
- **Catching `InterruptedException` and continuing.** When the JVM interrupts a thread, it's asking the thread to stop. Either let it propagate or call `Thread.currentThread.interrupt()` after catching it, so the signal isn't lost.
- **Returning `null` from a catch.** Turns one failure into a downstream `NullPointerException`. Return `Option`, `Try`, or `Either` instead.
- **Forgetting `close` in `try`/`finally`.** Use `Using` — guaranteed cleanup is the entire reason it exists.
- **Closing a resource that's about to be used asynchronously.** The resource will be gone by the time the async work touches it. Either keep the lifetime inside the `Future` block or move to `Resource` / `Scope`.
- **Using `Try` for domain errors.** If you keep pattern-matching on `Failure(_: MyException)`, that's an `Either[E, A]` trying to escape. Promote the failure to a named ADT.

## Putting it together — a small config loader

Tie the pieces together. Read a file, parse each line as `key=value`, return either a typed config map or a typed error.

In [ ]:
import scala.util.Using
import java.io.{BufferedReader, FileReader}

enum ConfigError:
  case NotFound(path: String)
  case Unreadable(path: String, cause: Throwable)
  case BadLine(path: String, line: String)

def loadConfig(path: String): Either[ConfigError, Map[String, String]] =
  val rawLines: Either[ConfigError, List[String]] =
    Using(new BufferedReader(new FileReader(path))) { reader =>
      Iterator.continually(reader.readLine())
        .takeWhile(_ != null)
        .map(_.trim)
        .filter(line => line.nonEmpty && !line.startsWith("#"))
        .toList
    }.toEither.left.map {
      case _: java.io.FileNotFoundException => ConfigError.NotFound(path)
      case other                            => ConfigError.Unreadable(path, other)
    }

  rawLines.flatMap { lines =>
    lines.foldLeft[Either[ConfigError, Map[String, String]]](Right(Map.empty)) {
      case (Right(acc), line) =>
        line.split("=", 2) match
          case Array(k, v) => Right(acc.updated(k.trim, v.trim))
          case _           => Left(ConfigError.BadLine(path, line))
      case (left, _) => left
    }
  }

Walk through what just happened. `Using` opens the file and guarantees the close. The body collects raw lines, dropping blanks and comments. `.toEither.left.map` translates the I/O throwable into a domain error — `NotFound` for missing files, `Unreadable` for everything else. Then `flatMap` runs the parsing step, which can fail per-line with `BadLine`. The fold short-circuits on the first `Left` and otherwise accumulates entries into the map.

The whole pipeline produces an `Either[ConfigError, Map[String, String]]` — typed input, typed failures, no exception escape from inside. This is the shape to aim for at the edges of every Scala program: errors named, resources released, the type signature telling the truth about what can go wrong.

## What's next

Notebook 15 returns to the motivating goal of the whole track: **Scala for Spark**. You'll see how the constructs you've learned — case classes, pattern matching, the collection combinators, `Option`/`Either`, futures, and the resource and error patterns from this notebook — combine to read Spark code idiomatically and write it without surprises. The Scala foundation ends here; the data-engineering payoff starts there.